In [1]:
import os
import re
import glob
import sqlite3
import pandas as pd

ROOT = r'D:\Bluestock work\bluestock-fintech-analysis1'
RAW_DIR = os.path.join(ROOT, 'data', 'raw')

for key in ['04_monthly_sip_inflows', '05_category_inflows', '06_industry_folio_count',
            '09_portfolio_holdings', '10_benchmark_indices']:
    path = glob.glob(os.path.join(RAW_DIR, '*' + key + '*'))[0]
    df = pd.read_csv(path)
    print(key, df.shape)
    print(df.dtypes)
    print()

04_monthly_sip_inflows (48, 6)
month                            str
sip_inflow_crore               int64
active_sip_accounts_crore    float64
new_sip_accounts_lakh        float64
sip_aum_lakh_crore           float64
yoy_growth_pct               float64
dtype: object

05_category_inflows (144, 3)
month                   str
category                str
net_inflow_crore    float64
dtype: object

06_industry_folio_count (21, 6)
month                      str
total_folios_crore     float64
equity_folios_crore    float64
debt_folios_crore      float64
hybrid_folios_crore    float64
others_folios_crore    float64
dtype: object

09_portfolio_holdings (322, 8)
amfi_code              int64
stock_symbol             str
stock_name               str
sector                   str
weight_pct           float64
market_value_cr      float64
current_price_inr    float64
portfolio_date           str
dtype: object

10_benchmark_indices (8050, 3)
date               str
index_name         str
close_value    f

In [2]:
line = '| `amfi_code` | INTEGER | PK |'
print(re.findall(r'^\| `(\w+)` \| (\w+) \|', line))

[('amfi_code', 'INTEGER')]


In [4]:
DB_PATH = os.path.join(ROOT, 'data', 'processed', 'bluestock_dw.db')
DOC_PATH = os.path.join(ROOT, 'reports', 'data_dictionary.md')

with open(DOC_PATH, 'r', encoding='utf-8') as f:
    doc = f.read()

conn = sqlite3.connect(DB_PATH)

TABLES = ['dim_date', 'dim_fund', 'fact_nav', 'fact_transactions', 'fact_performance', 'fact_aum']

for table in TABLES:
    block = doc.split('### ' + table + '\n')[1]
    block = block.split('\n#')[0]
    doc_types = dict(re.findall(r'^\| `(\w+)` \| (\w+) \|', block, flags=re.MULTILINE))

    info = pd.read_sql_query('PRAGMA table_info(' + table + ')', conn)
    db_types = dict(zip(info['name'], info['type']))

    not_in_doc = sorted(set(db_types) - set(doc_types))
    not_in_db = sorted(set(doc_types) - set(db_types))
    wrong_type = []
    for col in doc_types:
        if col in db_types and doc_types[col] != db_types[col]:
            wrong_type.append(col)

    print(table, '| not in doc:', not_in_doc, '| not in database:', not_in_db, '| wrong type:', wrong_type)

conn.close()

IndexError: list index out of range